# Data Cleaning - Capa SILVER

En este notebook se realiza la transformación y limpieza de los datos para la capa SILVER.

El objetivo es preparar la información proveniente de la capa BRONZE para su posterior análisis y explotación.

In [1]:
import os

# Configurar variables de entorno para Hadoop en Windows
HADOOP_HOME = os.environ.get("HADOOP_HOME", "C:\\hadoop")

os.environ["HADOOP_HOME"] = HADOOP_HOME
os.environ["hadoop.home.dir"] = HADOOP_HOME
os.environ["PATH"] = f"{os.environ.get('PATH', '')};{HADOOP_HOME}\\bin"

In [2]:
# Importar librerías

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when, trim, lower, regexp_replace, to_timestamp, coalesce, try_to_timestamp, lit

## Crear la sesión de Apache Spark

Se crea una sesión de Spark para el procesamiento distribuido de datos.

In [3]:
# Crear sesión de Spark

spark = (
    SparkSession.builder
    .appName("FinancialDigitalTwin_Silver")
    .getOrCreate()
)

print("Spark inicializado")

Spark inicializado


## Cargar los conjuntos de datos

In [4]:
# Cargar datasets

INPUT_PATH = "../data/processed/bronze"
OUTPUT_PATH = "../data/processed/silver"

# Crear directorio SILVER si no existe

import os

os.makedirs(OUTPUT_PATH, exist_ok=True)

users_spark = (
    spark.read
    .parquet(f"{INPUT_PATH}/users.parquet")
)

cards_spark = (
    spark.read
    .parquet(f"{INPUT_PATH}/cards.parquet")
)

transactions_spark = (
    spark.read
    .parquet(f"{INPUT_PATH}/transactions.parquet")
)

print("Datos cargados")

Datos cargados


## Explorar la información

In [5]:
# Mostrar primeras filas

users_spark.show(5)
cards_spark.show(5)
transactions_spark.show(5)

+----+-----------+--------------+----------+-----------+------+--------------------+--------+---------+-----------------+-------------+----------+------------+----------------+
|  id|current_age|retirement_age|birth_year|birth_month|gender|             address|latitude|longitude|per_capita_income|yearly_income|total_debt|credit_score|num_credit_cards|
+----+-----------+--------------+----------+-----------+------+--------------------+--------+---------+-----------------+-------------+----------+------------+----------------+
| 825|         53|            66|      1966|         11|female|       462 rose lane|   34.15|  -117.76|          29278.0|      59696.0|  127613.0|         787|               5|
|1746|         53|            68|      1966|         12|female|3606 federal boul...|   40.76|   -73.74|          37891.0|      77254.0|  191349.0|         701|               5|
|1718|         81|            67|      1938|         11|female|     766 third drive|   34.02|  -117.89|          22

In [6]:
# Mostrar esquema

users_spark.printSchema()
cards_spark.printSchema()
transactions_spark.printSchema()

root
 |-- id: integer (nullable = true)
 |-- current_age: integer (nullable = true)
 |-- retirement_age: integer (nullable = true)
 |-- birth_year: integer (nullable = true)
 |-- birth_month: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- address: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- per_capita_income: float (nullable = true)
 |-- yearly_income: float (nullable = true)
 |-- total_debt: float (nullable = true)
 |-- credit_score: integer (nullable = true)
 |-- num_credit_cards: integer (nullable = true)

root
 |-- id: integer (nullable = true)
 |-- client_id: integer (nullable = true)
 |-- card_brand: string (nullable = true)
 |-- card_type: string (nullable = true)
 |-- card_number: double (nullable = true)
 |-- expires: string (nullable = true)
 |-- cvv: integer (nullable = true)
 |-- has_chip: string (nullable = true)
 |-- num_cards_issued: integer (nullable = true)
 |-- credit_limit: doubl

## Transformar los datos

In [7]:
# Transformar users

users_silver = users_spark.withColumn(
    "gender",
    trim(lower(col("gender")))
)

users_silver = users_silver.withColumn(
    "address",
    trim(lower(col("address")))
)

users_silver = users_silver.withColumn(
    "yearly_income",
    col("yearly_income").cast("double")
)

users_silver = users_silver.withColumn(
    "per_capita_income",
    col("per_capita_income").cast("double")
)

users_silver = users_silver.withColumn(
    "total_debt",
    col("total_debt").cast("double")
)

In [8]:
# Transformar cards

cards_silver = cards_spark.withColumn(
    "card_brand",
    trim(lower(col("card_brand")))
)

cards_silver = cards_silver.withColumn(
    "card_type",
    trim(lower(col("card_type")))
)

cards_silver = cards_silver.withColumn(
    "has_chip",
    trim(lower(col("has_chip")))
)

cards_silver = cards_silver.withColumn(
    "card_on_dark_web",
    trim(lower(col("card_on_dark_web")))
)

cards_silver = cards_silver.withColumn(
    "credit_limit",
    col("credit_limit").cast("double")
)

In [9]:
# Transformar transactions

transactions_silver = transactions_spark.withColumn(
    "use_chip",
    trim(lower(col("use_chip")))
)

transactions_silver = transactions_silver.withColumn(
    "merchant_city",
    trim(lower(col("merchant_city")))
)

transactions_silver = transactions_silver.withColumn(
    "merchant_state",
    trim(lower(col("merchant_state")))
)

transactions_silver = transactions_silver.withColumn(
    "amount",
    col("amount").cast("double")
)

## Convertir fechas

In [10]:
# Convertir fecha de transactions

transactions_silver = transactions_silver.withColumn(
    "date",
    coalesce(
        try_to_timestamp(
            col("date"),
            lit("dd/MM/yyyy HH:mm")
        ),
        try_to_timestamp(
            col("date"),
            lit("MM/dd/yyyy HH:mm")
        )
    )
)

## Eliminar registros duplicados

In [11]:
# Eliminar duplicados

users_silver = users_silver.dropDuplicates(["id"])

cards_silver = cards_silver.dropDuplicates(["id"])

transactions_silver = transactions_silver.dropDuplicates(["id"])

## Verificar valores faltantes

In [12]:
# Verificar valores faltantes

users_nulls = users_silver.select([
    count(
        when(col(c).isNull(), c)
    ).alias(c)
    for c in users_silver.columns
])

cards_nulls = cards_silver.select([
    count(
        when(col(c).isNull(), c)
    ).alias(c)
    for c in cards_silver.columns
])

transactions_nulls = transactions_silver.select([
    count(
        when(col(c).isNull(), c)
    ).alias(c)
    for c in transactions_silver.columns
])

print("Valores faltantes en users:")
users_nulls.show()

print("Valores faltantes en cards:")
cards_nulls.show()

print("Valores faltantes en transactions:")
transactions_nulls.show()

Valores faltantes en users:
+---+-----------+--------------+----------+-----------+------+-------+--------+---------+-----------------+-------------+----------+------------+----------------+
| id|current_age|retirement_age|birth_year|birth_month|gender|address|latitude|longitude|per_capita_income|yearly_income|total_debt|credit_score|num_credit_cards|
+---+-----------+--------------+----------+-----------+------+-------+--------+---------+-----------------+-------------+----------+------------+----------------+
|  0|          0|             0|         0|          0|     0|      0|       1|        0|                0|            0|         0|           0|               0|
+---+-----------+--------------+----------+-----------+------+-------+--------+---------+-----------------+-------------+----------+------------+----------------+

Valores faltantes en cards:
+---+---------+----------+---------+-----------+-------+---+--------+----------------+------------+--------------+--------------

## Guardar SILVER

In [13]:
# Guardar SILVER

users_silver.write.mode("overwrite").parquet(
    f"{OUTPUT_PATH}/users.parquet"
)

cards_silver.write.mode("overwrite").parquet(
    f"{OUTPUT_PATH}/cards.parquet"
)

transactions_silver.write.mode("overwrite").parquet(
    f"{OUTPUT_PATH}/transactions.parquet"
)

print("Pipeline SILVER completado")

Pipeline SILVER completado


In [14]:
# Finalizar Spark

spark.stop()